# Démo d'exécution symbolique avec angr

Ce notebook illustre l'exécution symbolique d'un petit programme C avec la bibliothèque `angr`.

Plan :
- Rappel du programme C
- Chargement du binaire dans angr
- Définition d'entrées symboliques
- Exploration des chemins
- Extraction de cas de test concrets


## 1. Rappel du programme C (version originale)

Nous considérons le programme suivant :

```c
#include <stdio.h>

int main() {
    int x, y;
    scanf("%d %d", &x, &y);

    while (x > 0) {
        if (y > x) {
            y = y - x;
        } else if (y < x) {
            x = x - y;
        } else {
            x = x - 1;
            y = y - 1;
        }
    }

    printf("%d\n", y);
    return 0;
}
```


## 2. Import des bibliothèques et compilation du programme

Dans Binder, le code C est déjà présent dans `src/prog_original.c`.
Nous allons le compiler en un exécutable `prog_original`.

In [ ]:
!gcc -o prog_original ../src/prog_original.c
!ls -l prog_original

In [ ]:
import angr
import claripy


## 3. Chargement du binaire dans angr

On charge l'exécutable compilé dans un projet angr.

In [ ]:
project = angr.Project("prog_original", auto_load_libs=False)
project

## 4. Définition d'entrées symboliques

Le programme lit deux entiers `x` et `y` depuis l'entrée standard.
Nous allons modéliser cette entrée comme un buffer symbolique contenant deux entiers séparés par un espace.


In [ ]:
import claripy

# On crée un buffer symbolique de 8 octets (simplification : petits entiers)
input_size = 8
sym_stdin = claripy.BVS('sym_stdin', input_size * 8)

state = project.factory.full_init_state(stdin=sym_stdin)

simgr = project.factory.simgr(state)
simgr

## 5. Exploration symbolique

Nous demandons à angr d'explorer les chemins possibles jusqu'à la fin du programme.


In [ ]:
simgr.run()
simgr

## 6. Extraction de cas de test concrets

Pour quelques états terminaux, nous allons demander à angr de nous donner des valeurs concrètes pour l'entrée symbolique.


In [ ]:
for i, f in enumerate(simgr.deadended[:5]):
    concrete = f.posix.stdin.load(0, input_size)
    model = f.solver.eval(concrete, cast_to=bytes)
    print(f"Cas {i+1} : entrée brute =", model)


## 7. Variante : programme simplifié

On peut répéter la même démarche avec `prog_simplified.c` pour comparer le comportement.
Cette partie peut servir de prolongement si vous avez plus de temps en séance.
